## Install Libraries

In [ ]:
!pip install transformers peft torch gymnasium

In [ ]:
!pip install "trl<=0.11.0" --upgrade

In [ ]:
!pip install k8s-agent-sandbox-gymnasium --upgrade

## Example usage

In [ ]:
from k8s_agent_sandbox import SandboxClient
from k8s_agent_sandbox.models import SandboxGatewayConnectionConfig, SandboxInClusterConnectionConfig

config = SandboxInClusterConnectionConfig()
client = SandboxClient(connection_config=config)
sandbox = client.create_sandbox(warmpool="simple-sandbox-warmpool")
try:
    result = sandbox.commands.run("pwd && ls")
    print("stdout:", result.stdout)
    print("stderr:", result.stderr)
    print("exit_code:", result.exit_code)
finally:
    sandbox.terminate()

In [ ]:
from k8s_agent_sandbox import SandboxClient
from k8s_agent_sandbox.models import SandboxInClusterConnectionConfig

from k8s_agent_sandbox_gymnasium import SandboxEnv, SparseTaskReward, StepPenaltyReward, SparseTaskTermination

env = SandboxEnv(
    reward_fn=StepPenaltyReward(
        base=SparseTaskReward(
            success_fn=lambda obs, info: "manage.py" in obs and info["exit_code"] == 0
        ),
        penalty=0.02,
    ),
    termination_fn=SparseTaskTermination(
        success_fn=lambda obs, info: "manage.py" in obs and info["exit_code"] == 0
    ),
    warmpool="simple-sandbox-warmpool",
    client=SandboxClient(connection_config=SandboxInClusterConnectionConfig()),
    max_episode_steps=15,
)

agent_tasks = [
    "create a django poll project",
]

trajectory = []


actions = ["pwd", "ls -la", "touch manage.py"]
try:
    for task in agent_tasks:
        obs, info = env.reset(options={"task": task})
        terminated, truncated = False, False

        i = 0
        while not terminated and not truncated:
            action = actions[i] + "&& pwd && ls"
            i += 1
            obs, reward, terminated, truncated, info = env.step(action)
            print(f"""obs={obs}
reward={reward}
terminated={terminated}
truncated={truncated}
info={info}
-----------------------
""")
            trajectory.append((obs, action, reward, info))
finally:
    env.close()

## Fine-tune an LLM

In [ ]:
import torch
import os
from transformers import AutoTokenizer
from trl import AutoModelForCausalLMWithValueHead, PPOConfig, PPOTrainer
from peft import LoraConfig

from k8s_agent_sandbox import SandboxClient
from k8s_agent_sandbox.models import SandboxInClusterConnectionConfig

from k8s_agent_sandbox_gymnasium import SandboxEnv, SparseTaskReward, StepPenaltyReward, SparseTaskTermination

# ── 1. Configuration & Setup ─────────────────────────────────────────────────

model_name = "Qwen/Qwen2.5-Coder-1.5B"  # Good starting point for code/bash
ppo_config = PPOConfig(
    learning_rate=1.41e-5,
    batch_size=4,           # Trigger an update every 4 steps
    mini_batch_size=1,      # Must be a divisor of batch_size
    # gradient_accumulation_steps=4 # You can use this if VRAM is tight
)

# Apply LoRA to make training memory-efficient
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

# ── 2. Load Model and Tokenizer ──────────────────────────────────────────────

# TRL uses a special wrapper that adds a "Value Head" to the LLM for PPO
model = AutoModelForCausalLMWithValueHead.from_pretrained(
    model_name,
    peft_config=lora_config,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Initialize the PPO Trainer
ppo_trainer = PPOTrainer(
    config=ppo_config,
    model=model,
    ref_model=None, # TRL handles the reference model internally with PEFT
    tokenizer=tokenizer,
)

# ── 3. Initialize Your Custom Environment ────────────────────────────────────

# Let's use a task where the agent must create a specific file
env = SandboxEnv(
    reward_fn=SparseTaskReward(
        success_fn=lambda obs, info: "hello_world.sh" in obs and info["exit_code"] == 0
    ),
    termination_fn=SparseTaskTermination(
        success_fn=lambda obs, info: "hello_world.sh" in obs and info["exit_code"] == 0
    ),
    client=SandboxClient(connection_config=SandboxInClusterConnectionConfig()),
    warmpool="simple-sandbox-warmpool",
    max_episode_steps=5,
)

# ── 4. The Training Loop ─────────────────────────────────────────────────────

epochs = 5
task = "Create a bash script named hello_world.sh that prints 'Hello World'."

# 1. Global buffers to hold data across episodes
queries_buffer = []
responses_buffer = []
rewards_buffer = []

try:
    for epoch in range(epochs):
        obs, info = env.reset(options={"task": task})
        terminated, truncated = False, False
        
        print(f"\n--- Starting Epoch {epoch} ---")

        while not terminated and not truncated:
            prompt = f"""System: You are an expert bash programmer. You can create only bash commands: "sh -c \"<YOUR_COMMAND>\""
You need to solve this task: {task}
Terminal Output: {obs}"""
            query_tensor = tokenizer(prompt, return_tensors="pt").input_ids.squeeze().to(ppo_trainer.accelerator.device)
            
            generation_kwargs = {"max_new_tokens": 32, "do_sample": True, "top_k": 0.0, "top_p": 1.0}
            response_tensor = ppo_trainer.generate(query_tensor, **generation_kwargs).squeeze()
            
            # Only take the newly generated tokens
            action_tensor = response_tensor[len(query_tensor):]
            action = tokenizer.decode(action_tensor, skip_special_tokens=True).strip() + " && pwd && ls"

            next_obs, reward, terminated, truncated, info = env.step(action)
            print(f"""action={action}
obs={next_obs}
reward={reward}
terminated={terminated}
truncated={truncated}
info={info}
-----------------------
""")    
            # 2. Append the tensors to our global buffer
            queries_buffer.append(query_tensor)
            responses_buffer.append(action_tensor)
            rewards_buffer.append(torch.tensor(reward, dtype=torch.float32))
            
            # 3. Check if the buffer is full
            if len(queries_buffer) == ppo_config.batch_size:
                print(">> Buffer full! Executing PPO optimization step...")
                stats = ppo_trainer.step(queries_buffer, responses_buffer, rewards_buffer)
                print(f">> PPO Loss: {stats['ppo/loss/total']}")
                
                # Clear the buffer for the next batch
                queries_buffer = []
                responses_buffer = []
                rewards_buffer = []
            
            obs = next_obs
            
            if reward > 0:
                print("Goal reached! Ending episode early.")
                break
finally:
    env.close()

# Save your fine-tuned model adapters!
path = "./sandbox-agent-lora"
if not os.path.exists(path):
    os.makedirs(path)
model.save_pretrained(path)

## Load and Test the Fine-tuned LLM

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# ── 1. Configuration ─────────────────────────────────────────────────────────

base_model_name = "Qwen/Qwen2.5-Coder-1.5B"
adapter_path = "./sandbox-agent-lora"

# ── 2. Load Model and Tokenizer ──────────────────────────────────────────────

print("Loading base model...")
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
tokenizer.pad_token = tokenizer.eos_token

# Load the base model
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    device_map="auto",
    torch_dtype=torch.float16,  # Uses less VRAM during inference
)

print("Applying LoRA adapter...")
# Merge the fine-tuned adapter weights with the base model
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()  # Set model to evaluation mode

# ── 3. Prepare the Environment State ─────────────────────────────────────────

task = "Create a bash script named hello_world.sh that prints 'Hello World'."
# Assuming this is step 1, the initial observation from env.reset() is:
obs = "Sandbox ready." 

# CRITICAL: The prompt format must perfectly match how it was trained
prompt = f"""System: You are an expert sh programmer. 
You run commands only in this format: sh -c \"<YOUR TARGET COMMAND> && ls\".
Task: {task}
Terminal Output: {obs}
Command to run:"""

# ── 4. Generate the Action ───────────────────────────────────────────────────

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

print("Generating command...")
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=32,
        do_sample=False,  # Greedy decoding ensures the most confident, deterministic output
        pad_token_id=tokenizer.eos_token_id
    )

# ── 5. Extract and Display the Command ───────────────────────────────────────

# Slice the output tensor to ignore the prompt and only decode the newly generated tokens
input_length = inputs.input_ids.shape[1]
generated_tokens = outputs[0][input_length:]

action = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

print("\n" + "="*40)
print(f"Prompt Context:\n{prompt}")
print("="*40)
print(f"Agent's Command: {action}")
print("="*40)